In [ ]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [28]:


spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.dm_policy_retention
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/dm_policy_retention'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.fact_policies
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_policies'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.dm_claims_experience
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/dm_claims_experience'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.star_members
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/star_members'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.dm_member_value
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/dm_member_value'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.ft_policy_churn_split
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_policy_churn_split'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.ft_claims_risk_split
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_claims_risk_split'
""")





DataFrame[]

# Cell 1 – Setup

In [13]:
# Dashboard Views – Setup

from pyspark.sql import functions as F

DB_GOLD = "bupa_gold"

print("Using Gold database:", DB_GOLD)
spark.sql(f"USE {DB_GOLD}")
spark.sql("SHOW TABLES").show(truncate=False)


Using Gold database: bupa_gold
+---------+--------------------+-----------+
|namespace|tableName           |isTemporary|
+---------+--------------------+-----------+
|bupa_gold|dm_claims_experience|false      |
|bupa_gold|dm_policy_retention |false      |
|bupa_gold|fact_policies       |false      |
|bupa_gold|vw_policy_portfolio |false      |
+---------+--------------------+-----------+



# Cell 2 – Policy portfolio view (detail from fact_policies)

In [14]:
# Cell 2 — Policy portfolio view (detail-level, 1 row per policy)

spark.sql(f"""
CREATE OR REPLACE VIEW {DB_GOLD}.vw_policy_portfolio AS
SELECT
    Policy_ID,
    Customer_ID,
    Product_Line,
    Channel,
    Sum_Insured_GBP,
    Annual_Premium_GBP,
    Policy_Start_Date,
    Policy_End_Date,
    Policy_Duration_Days,
    Tenure_Band,
    Premium_Band,
    Discount_Band,
    Renewal_Offered_Flag,
    Renewal_Accepted_Flag,
    Renewal_Conversion,
    Renewal_Outcome,
    dq_money_valid,
    dq_discount_valid,
    dq_renewal_valid,
    created_at,
    source_system,
    record_hash
FROM {DB_GOLD}.fact_policies
""")

print("✅ Created view:", f"{DB_GOLD}.vw_policy_portfolio")
spark.table(f"{DB_GOLD}.vw_policy_portfolio").show(10, truncate=False)
spark.table(f"{DB_GOLD}.vw_policy_portfolio").printSchema()


✅ Created view: bupa_gold.vw_policy_portfolio


+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+-----------+-----------------+-------------+--------------------+---------------------+------------------+---------------+--------------+-----------------+----------------+--------------------------+----------------------------+----------------------------------------------------------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Renewal_Outcome|dq_money_valid|dq_discount_valid|dq_renewal_valid|created_at                |source_system               |record_hash                                                     |
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+----------------

Usage in Power BI / Tableau

Use this view for portfolio-level dashboards:

Premium by product/channel

Tenure distribution

Renewal outcome breakdown

Discount vs renewal conversion, etc.

# Cell 3 – Policy retention KPI view (from dm_policy_retention)

In [15]:
# Cell 3 — Policy retention KPI view (aggregated)

spark.sql(f"""
CREATE OR REPLACE VIEW {DB_GOLD}.vw_policy_retention_kpi AS
SELECT
    Product_Line,
    Channel,
    Tenure_Band,
    Policy_Start_Year,
    Policy_Count,
    Total_Premium_GBP,
    Avg_Premium_GBP,
    Offer_Rate,
    Acceptance_Rate,
    Clean_Renewal_Conversion
FROM {DB_GOLD}.dm_policy_retention
""")

print("✅ Created view:", f"{DB_GOLD}.vw_policy_retention_kpi")
spark.table(f"{DB_GOLD}.vw_policy_retention_kpi").show(10, truncate=False)
spark.table(f"{DB_GOLD}.vw_policy_retention_kpi").printSchema()


✅ Created view: bupa_gold.vw_policy_retention_kpi
+------------+-------+-----------+-----------------+------------+-----------------+---------------+----------+---------------+------------------------+
|Product_Line|Channel|Tenure_Band|Policy_Start_Year|Policy_Count|Total_Premium_GBP|Avg_Premium_GBP|Offer_Rate|Acceptance_Rate|Clean_Renewal_Conversion|
+------------+-------+-----------+-----------------+------------+-----------------+---------------+----------+---------------+------------------------+
|Dental      |Broker |6–12 months|2024             |10          |216588.0         |21658.8        |1.0       |0.7            |0.7                     |
|Dental      |Partner|1–2 years  |2020             |832         |2.5988647E7      |31236.35       |0.899     |0.6238         |0.6939                  |
|Dental      |Online |6–12 months|2023             |25          |65750.0          |2630.0         |1.0       |0.84           |0.84                    |
|Dental      |Partner|6–12 months|2021

Use this for:

High-level KPI tiles and trend charts
(e.g. acceptance rate by year, conversion by channel & tenure band).

# Cell 4 – Member profile & value views

In [20]:
# Cell 4 — Member profile (star_members) and member value (dm_member_value)

# 4a) Member profile – detailed, 1 row per member
spark.sql(f"""
CREATE OR REPLACE VIEW {DB_GOLD}.vw_member_profile AS
SELECT
    Member_ID,
    Policy_ID,
    Age,
    Gender,
    BMI,
    Chronic_Disease,
    Employment_Status,
    Region_Code,
    dq_age_valid,
    dq_bmi_valid
FROM {DB_GOLD}.star_members
""")

print("✅ Created view:", f"{DB_GOLD}.vw_member_profile")
spark.table(f"{DB_GOLD}.vw_member_profile").show(10, truncate=False)
spark.table(f"{DB_GOLD}.vw_member_profile").printSchema()


# 4b) Member value – aggregated per segment, etc. (from dm_member_value)

spark.sql(f"""
CREATE OR REPLACE VIEW {DB_GOLD}.vw_member_value_kpi AS
SELECT
    *
FROM {DB_GOLD}.dm_member_value
""")

print("✅ Created view:", f"{DB_GOLD}.vw_member_value_kpi")
spark.table(f"{DB_GOLD}.vw_member_value_kpi").show(10, truncate=False)
spark.table(f"{DB_GOLD}.vw_member_value_kpi").printSchema()


✅ Created view: bupa_gold.vw_member_profile


+------------+----------+---+------+------------------+---------------+-----------------+-----------+------------+------------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Chronic_Disease|Employment_Status|Region_Code|dq_age_valid|dq_bmi_valid|
+------------+----------+---+------+------------------+---------------+-----------------+-----------+------------+------------+
|MEM_00000008|P_00000008|56 |F     |23.140940890221643|Asthma         |Student          |280        |1           |1           |
|MEM_00001714|P_00001714|71 |M     |27.963719840043765|None           |Employed         |360        |1           |1           |
|MEM_00001746|P_00001746|75 |M     |30.541100957674118|Diabetes       |Student          |80         |1           |1           |
|MEM_00001967|P_00001967|37 |F     |25.982850362358867|Hypertension   |Employed         |280        |1           |1           |
|MEM_00001995|P_00001995|66 |M     |32.6294573451364  |Asthma         |Employed         |360        |1  

+----------+------------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+---------------+--------------+-----------------+----------------+--------------------------+----------------------------+----------------------------------------------------------------+------------+----------------+----------------------+---------------+---------------+--------------------+-----------------+---------------+--------+
|Policy_ID |Member_ID   |Age|Gender|BMI               |Smoker_Flag|Chronic_Disease|Employment_Status|Region|Age_Band|BMI_Band  |Smoker_Status|Chronic_Flag|Chronic_Group|Employment_Group|Region_Group  |dq

Use:

vw_member_profile → demographic / health profile dashboards

vw_member_value_kpi → segment-level LTV / claim cost / premium KPIs.

# Cell 5 – Claims experience view (from dm_claims_experience)

In [21]:
# Cell 5 — Claims experience KPIs (aggregated)

spark.sql(f"""
CREATE OR REPLACE VIEW {DB_GOLD}.vw_claims_experience_kpi AS
SELECT
    *
FROM {DB_GOLD}.dm_claims_experience
""")

print("✅ Created view:", f"{DB_GOLD}.vw_claims_experience_kpi")
spark.table(f"{DB_GOLD}.vw_claims_experience_kpi").show(10, truncate=False)
spark.table(f"{DB_GOLD}.vw_claims_experience_kpi").printSchema()


✅ Created view: bupa_gold.vw_claims_experience_kpi


+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+------------------+------------------+--------------+---------+-------------------+----------------+-----------------------+
|Claim_ID |Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|Claim_Amount_GBP  |Fraud_Label|Claim_Type|Claim_Status|Provider_Fraud_Flag|Days_To_Settle|Payout_to_Amount_Ratio|High_Cost_Claim_Flag|dq_money_valid|dq_date_valid|Exposure_GBP      |Payout_Ratio      |High_Cost_Flag|Cost_Band|Settlement_SLA_Band|Open_Closed_Flag|Provider_Fraud_Flag_Ref|
+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+------------------+------------------+-------

Use this for:

Loss ratios by claim type / status / provider risk tier

High-cost vs normal claim mix

Fraud-labelled claim rates, etc.

# Cell 6 – ML: policy churn feature view

This doesn’t depend on the trained model; it just exposes the ML feature table, which is exactly what a data scientist or ML Ops engineer would connect to.

In [26]:
# Cell 6 — ML policy churn feature view (from ft_policy_churn_split)

spark.sql(f"""
CREATE OR REPLACE VIEW {DB_GOLD}.vw_ml_policy_churn_features AS
SELECT
    Policy_ID,
    Customer_ID,
    Product_Line,
    Channel,
    Sum_Insured_GBP,
    Annual_Premium_GBP,
    Policy_Start_Date,
    Policy_End_Date,
    Policy_Duration_Days,
    Renewal_Offered_Flag,
    Renewal_Accepted_Flag,
    Renewal_Conversion,
    Tenure_Band,
    Premium_Band,
    Discount_Band,
    dq_money_valid,
    dq_discount_valid,
    dq_renewal_valid,
    Churn_Label,
    Is_Discounted,
    Premium_per_1k_SumInsured,
    dataset_split
FROM {DB_GOLD}.ft_policy_churn_split
""")

print("✅ Created view:", f"{DB_GOLD}.vw_ml_policy_churn_features")
spark.table(f"{DB_GOLD}.vw_ml_policy_churn_features").groupBy("dataset_split", "Churn_Label").count().show()
spark.table(f"{DB_GOLD}.vw_ml_policy_churn_features").show(10, truncate=False)


✅ Created view: bupa_gold.vw_ml_policy_churn_features


+-------------+-----------+------+
|dataset_split|Churn_Label| count|
+-------------+-----------+------+
|        train|          0|159495|
|         test|          0| 39820|
|        train|          1| 68818|
|        train|       NULL| 76429|
|         test|          1| 17306|
|         test|       NULL| 19241|
+-------------+-----------+------+



+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+--------------+-----------------+----------------+-----------+-------------+-------------------------+-------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Tenure_Band|Premium_Band     |Discount_Band|dq_money_valid|dq_discount_valid|dq_renewal_valid|Churn_Label|Is_Discounted|Premium_per_1k_SumInsured|dataset_split|
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+--------------+----------------

Use for:

Monitoring label balance train vs test

Model input distributions (premium, tenure, discount behaviour)

Drift checks later.

# Cell 7 – ML: claims risk feature view

In [29]:
# Cell 7 — ML claims risk feature view (from ft_claims_risk_split)

spark.sql(f"""
CREATE OR REPLACE VIEW {DB_GOLD}.vw_ml_claims_risk_features AS
SELECT
    Claim_ID,
    Member_Key,
    Provider_ID,
    Claim_Type_Name,
    Claim_Type_Code,
    Claim_Status,
    Claim_Amount_GBP,
    Payout_GBP,
    Payout_to_Amount_Ratio,
    High_Cost_Claim_Flag,
    Claim_Fraud_Label,
    Provider_Fraud_Label,
    Provider_Risk_Tier,
    dq_money_valid,
    dq_date_valid,
    Is_Fraudulent_Claim,
    Is_High_Cost,
    dataset_split
FROM {DB_GOLD}.ft_claims_risk_split
""")

print("✅ Created view:", f"{DB_GOLD}.vw_ml_claims_risk_features")
spark.table(f"{DB_GOLD}.vw_ml_claims_risk_features").groupBy("dataset_split", "Is_Fraudulent_Claim").count().show()
spark.table(f"{DB_GOLD}.vw_ml_claims_risk_features").show(10, truncate=False)


✅ Created view: bupa_gold.vw_ml_claims_risk_features


+-------------+-------------------+------+
|dataset_split|Is_Fraudulent_Claim| count|
+-------------+-------------------+------+
|        train|                  0|283635|
|         test|                  0| 71276|
|        train|                  1|162467|
|         test|                  1| 40833|
+-------------+-------------------+------+



+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Claim_Type_Name|Claim_Type_Code|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|High_Cost_Claim_Flag|Claim_Fraud_Label|Provider_Fraud_Label|Provider_Risk_Tier|dq_money_valid|dq_date_valid|Is_Fraudulent_Claim|Is_High_Cost|dataset_split|
+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|CLM110013|BENE15435 |PRV51790   |Travel         |TRAVEL         |Settled     |244.07511199775644|200.0     |0.8194198841618822    |0   

Use for:

Monitoring class balance (fraud / high-cost) train vs test

Feature distribution by claim type / provider risk tier.

In [30]:
from pyspark.sql import functions as F

DB_GOLD = "bupa_gold"
spark.sql(f"USE {DB_GOLD}")

# ADLS config
GOLD_ACCOUNT   = "clientdatastorage"
GOLD_CONTAINER = "golddata"

def gold_path(name: str) -> str:
    return f"abfss://{GOLD_CONTAINER}@{GOLD_ACCOUNT}.dfs.core.windows.net/{name}"

paths_views = {
    "vw_policy_portfolio"         : gold_path("vw_policy_portfolio"),
    "vw_policy_retention_kpi"     : gold_path("vw_policy_retention_kpi"),
    "vw_member_profile"           : gold_path("vw_member_profile"),
    "vw_member_value_kpi"         : gold_path("vw_member_value_kpi"),
    "vw_claims_experience_kpi"    : gold_path("vw_claims_experience_kpi"),
    "vw_ml_policy_churn_features" : gold_path("vw_ml_policy_churn_features"),
    "vw_ml_claims_risk_features"  : gold_path("vw_ml_claims_risk_features"),
}

print("Materialisation targets:")
for k, v in paths_views.items():
    print(f"  {k:30s} -> {v}")


Materialisation targets:
  vw_policy_portfolio            -> abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_portfolio
  vw_policy_retention_kpi        -> abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_retention_kpi
  vw_member_profile              -> abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_member_profile
  vw_member_value_kpi            -> abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_member_value_kpi
  vw_claims_experience_kpi       -> abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_claims_experience_kpi
  vw_ml_policy_churn_features    -> abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_ml_policy_churn_features
  vw_ml_claims_risk_features     -> abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_ml_claims_risk_features


# Cell 2 – Helper to write a view to ADLS

In [31]:
def materialise_view(view_name: str, path: str):
    """
    Read a Spark SQL view from bupa_gold and write it as a Delta dataset to ADLS.
    Overwrites if it already exists.
    """
    print(f"\n▶ Materialising view {DB_GOLD}.{view_name} → {path}")
    
    df = spark.table(f"{DB_GOLD}.{view_name}")
    print("  Rowcount:", df.count())
    
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(path))
    
    print("  ✅ Written as Delta:", path)


# Cell 3 – Materialise all 7 views

In [32]:
# Policy views
materialise_view("vw_policy_portfolio",         paths_views["vw_policy_portfolio"])
materialise_view("vw_policy_retention_kpi",     paths_views["vw_policy_retention_kpi"])

# Member views
materialise_view("vw_member_profile",           paths_views["vw_member_profile"])
materialise_view("vw_member_value_kpi",         paths_views["vw_member_value_kpi"])

# Claims views
materialise_view("vw_claims_experience_kpi",    paths_views["vw_claims_experience_kpi"])

# ML feature views
materialise_view("vw_ml_policy_churn_features", paths_views["vw_ml_policy_churn_features"])
materialise_view("vw_ml_claims_risk_features",  paths_views["vw_ml_claims_risk_features"])



▶ Materialising view bupa_gold.vw_policy_portfolio → abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_portfolio
  Rowcount: 381109


25/12/07 23:33:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


  ✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_portfolio

▶ Materialising view bupa_gold.vw_policy_retention_kpi → abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_retention_kpi
  Rowcount: 183


  ✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_retention_kpi

▶ Materialising view bupa_gold.vw_member_profile → abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_member_profile
  Rowcount: 381109


  ✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_member_profile

▶ Materialising view bupa_gold.vw_member_value_kpi → abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_member_value_kpi
  Rowcount: 381109


...

  ✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_member_value_kpi

▶ Materialising view bupa_gold.vw_claims_experience_kpi → abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_claims_experience_kpi
  Rowcount: 558211


  ✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_claims_experience_kpi

▶ Materialising view bupa_gold.vw_ml_policy_churn_features → abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_ml_policy_churn_features
  Rowcount: 381109


25/12/07 23:34:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


  ✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_ml_policy_churn_features

▶ Materialising view bupa_gold.vw_ml_claims_risk_features → abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_ml_claims_risk_features
  Rowcount: 558211


  ✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_ml_claims_risk_features


You should see prints like:

Rowcount: ...

✅ Written as Delta: abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_portfolio

etc.

# (Optional) Cell 4 – Register external tables on top of ADLS paths

If you also want physical tables on top of those paths (not just views), you can add:

In [33]:
for view_name, path in paths_views.items():
    table_name = f"{view_name}_tbl"   # e.g. vw_policy_portfolio_tbl
    full_table = f"{DB_GOLD}.{table_name}"
    
    print(f"\n▶ Registering external table {full_table} on {path}")
    
    spark.sql(f"DROP TABLE IF EXISTS {full_table}")
    spark.sql(f"""
    CREATE TABLE {full_table}
    USING DELTA
    LOCATION '{path}'
    """)
    
    print("  ✅ Registered:", full_table)
    spark.table(full_table).show(3, truncate=False)



▶ Registering external table bupa_gold.vw_policy_portfolio_tbl on abfss://golddata@clientdatastorage.dfs.core.windows.net/vw_policy_portfolio
  ✅ Registered: bupa_gold.vw_policy_portfolio_tbl


+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+-----------+-----------------+-------------+--------------------+---------------------+------------------+---------------+--------------+-----------------+----------------+--------------------------+----------------------------+----------------------------------------------------------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Renewal_Outcome|dq_money_valid|dq_discount_valid|dq_renewal_valid|created_at                |source_system               |record_hash                                                     |
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+----------------

+------------+----------+---+------+------------------+---------------+-----------------+-----------+------------+------------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Chronic_Disease|Employment_Status|Region_Code|dq_age_valid|dq_bmi_valid|
+------------+----------+---+------+------------------+---------------+-----------------+-----------+------------+------------+
|MEM_00000008|P_00000008|56 |F     |23.140940890221643|Asthma         |Student          |280        |1           |1           |
|MEM_00001714|P_00001714|71 |M     |27.963719840043765|None           |Employed         |360        |1           |1           |
|MEM_00001746|P_00001746|75 |M     |30.541100957674118|Diabetes       |Student          |80         |1           |1           |
+------------+----------+---+------+------------------+---------------+-----------------+-----------+------------+------------+
only showing top 3 rows


▶ Registering external table bupa_gold.vw_member_value_kpi_tbl on abfss://gold

+----------+------------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+---------------+--------------+-----------------+----------------+--------------------------+----------------------------+----------------------------------------------------------------+------------+----------------+----------------------+---------------+---------------+--------------------+-----------------+---------------+--------+
|Policy_ID |Member_ID   |Age|Gender|BMI               |Smoker_Flag|Chronic_Disease|Employment_Status|Region|Age_Band|BMI_Band  |Smoker_Status|Chronic_Flag|Chronic_Group|Employment_Group|Region_Group  |dq

+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+------------------+------------------+--------------+---------+-------------------+----------------+-----------------------+
|Claim_ID |Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|Claim_Amount_GBP  |Fraud_Label|Claim_Type|Claim_Status|Provider_Fraud_Flag|Days_To_Settle|Payout_to_Amount_Ratio|High_Cost_Claim_Flag|dq_money_valid|dq_date_valid|Exposure_GBP      |Payout_Ratio      |High_Cost_Flag|Cost_Band|Settlement_SLA_Band|Open_Closed_Flag|Provider_Fraud_Flag_Ref|
+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+------------------+------------------+-------

+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+--------------+-----------------+----------------+-----------+-------------+-------------------------+-------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Tenure_Band|Premium_Band     |Discount_Band|dq_money_valid|dq_discount_valid|dq_renewal_valid|Churn_Label|Is_Discounted|Premium_per_1k_SumInsured|dataset_split|
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+--------------+----------------

+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Claim_Type_Name|Claim_Type_Code|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|High_Cost_Claim_Flag|Claim_Fraud_Label|Provider_Fraud_Label|Provider_Risk_Tier|dq_money_valid|dq_date_valid|Is_Fraudulent_Claim|Is_High_Cost|dataset_split|
+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|CLM110013|BENE15435 |PRV51790   |Travel         |TRAVEL         |Settled     |244.07511199775644|200.0     |0.8194198841618822    |0   